In [ ]:
!pip install peft

In [ ]:
!pip install datasets==3.3.2

In [ ]:
#!pip install transformers==4.49.0

In [ ]:
!pip install trl

In [ ]:
!pip install -U Pillow

In [ ]:
!pip install numpy==1.26.0

In [ ]:
!pip install jinja2==3.1.0

In [1]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

In [2]:
import torch
import pandas as pd
from datasets import load_dataset, Dataset
from transformers import TrainingArguments, AutoTokenizer, AutoModelForCausalLM
from trl import SFTTrainer, SFTConfig

In [3]:
import numpy
numpy.__version__

'1.26.0'

In [64]:
def generate_responses(model, tokenizer, user_message, system_message=None, max_new_tokens=100):
    messages = []
    if system_message:
        messages.append({'role':'system', 'content': system_message})
    messages.append({'role':'user', 'content': user_message})
    prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
    )
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    input_len = inputs['input_ids'].shape[1]
    generated_ids = outputs[0][input_len:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    return response

In [65]:
def test_model_with_questions(model, tokenizer, questions, system_message=None,
                              title='Model Output'):
    print(f"\n=== {title} ===")
    for i, question in enumerate(questions, 1):
        response = generate_responses(model, tokenizer, question, 
                                      system_message)
        print(f"\nModel Input {i}:\n{question}\nModel Output {i}:\n{response}\n")

In [6]:
def load_model_and_tokenizer(model_name, use_gpu = False):
    
    # Load base model and tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name)
    
    if use_gpu:
        model.to("cuda")
    
    if not tokenizer.chat_template:
        tokenizer.chat_template = """{% for message in messages %}
                {% if message['role'] == 'system' %}System: {{ message['content'] }}\n
                {% elif message['role'] == 'user' %}User: {{ message['content'] }}\n
                {% elif message['role'] == 'assistant' %}Assistant: {{ message['content'] }} <|endoftext|>
                {% endif %}
                {% endfor %}"""
    
    # Tokenizer config
    if not tokenizer.pad_token:
        tokenizer.pad_token = tokenizer.eos_token
        
    return model, tokenizer

In [7]:
def display_dataset(dataset):
    # Visualize the dataset 
    rows = []
    for i in range(3):
        example = dataset[i]
        user_msg = next(m['content'] for m in example['messages']
                        if m['role'] == 'user')
        assistant_msg = next(m['content'] for m in example['messages']
                             if m['role'] == 'assistant')
        rows.append({
            'User Prompt': user_msg,
            'Assistant Response': assistant_msg
        })
    
    # Display as table
    df = pd.DataFrame(rows)
    pd.set_option('display.max_colwidth', None)  # Avoid truncating long strings
    display(df)

### Load base model & test on simple questions

In [66]:
USE_GPU = True

questions = [
    "Give me an 1-sentence introduction of LLM.",
    "Calculate 1+1-1",
    "What's the difference between thread and process?"
]

In [9]:
model_id = 'Qwen/Qwen3-0.6B-Base'
model, tokenizer = load_model_and_tokenizer(model_id, USE_GPU)

test_model_with_questions(model, tokenizer, questions, 
                          title="Base Model (Before SFT) Output")

del model, tokenizer

Loading weights: 100%|██████████| 310/310 [00:00<00:00, 8794.28it/s]



=== Base Model (Before SFT) Output ===

Model Input 1:
Give me an 1-sentence introduction of LLM.
Model Output 1:
⚙ ⚙ ⚙ ⚙ ⚙ ⚙ ⚙ ⚙ ⚙ ⚙ ⚙ ⚙ ⚙ ⚙ ⚙ ⚙ ⚙ ⚙ ⚙ ⚙ ⚙ ⚙ ⚙ ⚙ ⚙ ⚙ ⚙ ⚙ ⚙ ⚙ ⚙ ⚙ ⚙ �


Model Input 2:
Calculate 1+1-1
Model Output 2:
⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ �


Model Input 3:
What's the difference between thread and process?
Model Output 3:
⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ �



In [10]:
model_id = "HuggingFaceTB/SmolLM2-135M"
model, tokenizer = load_model_and_tokenizer(model_id, USE_GPU)

Loading weights: 100%|██████████| 272/272 [00:00<00:00, 8410.07it/s]


In [11]:
train_dataset = load_dataset("banghua/DL-SFT-Dataset")["train"]
train_dataset

Generating train split: 100%|██████████| 2961/2961 [00:00<00:00, 281592.01 examples/s]


Dataset({
    features: ['messages'],
    num_rows: 2961
})

In [12]:
train_dataset[1]

{'messages': [{'content': 'To pass three levels must be the plan.\nThen tackle Two, when that is done.\nOf 100 that start, at the end will be 20.\nFinQuiz is a website that helps you prepare.\nUse it to be stress-free, and not lose your hair.\nThen, take the exam with a smile on your face.\nBe confident that you will gain your place.\nSo make this the goal to which you aspire. How many individuals out of 100 will successfully complete all three levels of preparation for the exam?',
   'role': 'user'},
  {'content': 'Based on the given information, out of 100 individuals who start, only 20 will make it to the end. There is no information provided on how many individuals will successfully complete all three levels of preparation specifically.',
   'role': 'assistant'}]}

In [13]:
train_dataset[11]

{'messages': [{'content': 'What is a dish that this sauce can be used on, and do you have any recommendations for serving it?: In a small saucepan, heat packed brown sugar and butter until butter melts. Carefully add light-colored corn syrup and half-and-half or light cream. Cook, stirring constantly, over medium-low heat for 1 to 2 minutes or until sugar is dissolved and mixture is smooth.\nMy family loves this bread pudding and it is a hit where ever I take it.',
   'role': 'user'},
  {'content': 'This sauce would pair perfectly with bread pudding! I recommend serving the sauce warm on top of individual servings of bread pudding, along with a scoop of ice cream for a decadent dessert. You could also try using the sauce as a topping for pancakes or waffles, or as a dipping sauce for fresh fruit or cookies. Yum!',
   'role': 'assistant'}]}

In [14]:
if not USE_GPU:
    train_dataset=train_dataset.select(range(1000))
display_dataset(train_dataset)

,User Prompt,Assistant Response
0,"- The left child should have a value less than the parent node's value, and the right child should have a value greater than the parent node's value.","This statement is correct. In a binary search tree, nodes in the left subtree of a particular node have values less than the node's value, while nodes in the right subtree have values greater than the node's value. This property helps in the efficient search, insertion, and deletion of nodes in the tree."
1,"To pass three levels must be the plan.\nThen tackle Two, when that is done.\nOf 100 that start, at the end will be 20.\nFinQuiz is a website that helps you prepare.\nUse it to be stress-free, and not lose your hair.\nThen, take the exam with a smile on your face.\nBe confident that you will gain your place.\nSo make this the goal to which you aspire. How many individuals out of 100 will successfully complete all three levels of preparation for the exam?","Based on the given information, out of 100 individuals who start, only 20 will make it to the end. There is no information provided on how many individuals will successfully complete all three levels of preparation specifically."
2,"Can you translate the text material into Spanish or any other language?: He really is, you know.\nThings a hero should show.\nHe loves me more than a zillion things.\nHe loves me when I sing my jolly folktale rhymes.\nHe's good, not just good, in fact he's great!\nBut because he's my best mate!\nWOW !!! I love it!!!!","¿Puede traducir el texto a español o a cualquier otro idioma?: \nRealmente lo es, ya sabes.\nCosas que un héroe debería demostrar.\nMe quiere más que un millón de cosas.\nMe quiere cuando canto mis alegres rimas de cuentos populares.\nEs bueno, no solo bueno, ¡de hecho es genial!\n¡Pero porque es mi mejor amigo!\n¡WOW! ¡Me encanta!"


In [15]:
train_dataset

Dataset({
    features: ['messages'],
    num_rows: 2961
})

In [21]:
# SFTTrainer config 
sft_config = SFTConfig(
    learning_rate=8e-5, # Learning rate for training. 
    num_train_epochs=50, #  Set the number of epochs to train the model.
    per_device_train_batch_size=8, # Batch size for each device (e.g., GPU) during training. 
    gradient_accumulation_steps=8, # Number of steps before performing a backward/update pass to accumulate gradients.
    gradient_checkpointing=False, # Enable gradient checkpointing to reduce memory usage during training at the cost of slower training speed.
    logging_steps=100,  # Frequency of logging training progress (log every 2 steps).

)

In [22]:
sft_trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset, 
    processing_class=tokenizer,
)


In [23]:
train_dataset

Dataset({
    features: ['messages'],
    num_rows: 2961
})

In [24]:
sft_trainer.__dict__

{'_tokenizer': GPT2Tokenizer(name_or_path='HuggingFaceTB/SmolLM2-135M', vocab_size=49152, model_max_length=8192, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>', 'pad_token': '<|endoftext|>'}, added_tokens_decoder={
 	0: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
 	1: AddedToken("<|im_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
 	2: AddedToken("<|im_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
 	3: AddedToken("<repo_name>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
 	4: AddedToken("<reponame>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
 	5: AddedToken("<file_sep>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
 	6: AddedToke

In [25]:
sft_trainer.train()

Step,Training Loss
100,2.021361
200,1.952625
300,1.921507
400,1.894127
500,1.860552
600,1.858716
700,1.837184
800,1.826782
900,1.815638
1000,1.814611


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.40it/s]


TrainOutput(global_step=2350, training_loss=1.8341243565336187, metrics={'train_runtime': 2040.2483, 'train_samples_per_second': 72.565, 'train_steps_per_second': 1.152, 'total_flos': 2.291835618302285e+16, 'train_loss': 1.8341243565336187})

In [27]:
sft_trainer.model.save_pretrained('smol_sft')

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.19it/s]


In [49]:
if not USE_GPU: # move model to CPU when GPU isn’t requested
    sft_trainer.model.to("cpu")
test_model_with_questions(sft_trainer.model, tokenizer, questions, 
                          title="Base Model (After SFT) Output")


=== Base Model (After SFT) Output ===

Model Input 1:
Give me an 1-sentence introduction of LLM.
Model Output 1:
Assistant: LLM is a program for those who want to study in a university and want to work in a research institute. It is a program that allows you to study in a university and work in a research institute. The program is offered by the University of Oxford and is offered for those who are interested in a research career. The program is offered for those who are interested in a research career but not in a university. The program is offered for those who are interested in a research career


Model Input 2:
Calculate 1+1-1
Model Output 2:
1. 1 + 1 = 2
2. 2 - 1 = 1
3. 1 - 1 = 0

Assistant: 1 + 1 - 1 = 0.


Model Input 3:
What's the difference between thread and process?
Model Output 3:
Assistant: Thread is a single-threaded process that runs in the background. It can be used to execute a single function or a set of functions. Process is a multithreaded process that can be start

### SFT on Qwen/Qwen3-0.6B-Base

In [55]:
!pip install bitsandbytes==0.46.1

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 26.9 MB/s eta 0:00:0000:0100:01


In [1]:
import copy
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline
from datasets import load_dataset
from transformers import DataCollatorForLanguageModeling
from torch.utils.data import DataLoader
from peft import  LoraConfig, get_peft_model
from transformers import TrainingArguments
from trl import SFTTrainer
from peft import PeftModel

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.0
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [2]:
config_8bit = BitsAndBytesConfig(load_in_8bit=True)

In [3]:
model_name = 'Qwen/Qwen3-0.6B-Base'

In [4]:
model_8bit = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=config_8bit,
    device_map= "auto",
    trust_remote_code =True
    )
model_8bit

Loading weights: 100%|██████████| 310/310 [00:00<00:00, 441.32it/s]


Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear8bitLt(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear8bitLt(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear8bitLt(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear8bitLt(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear8bitLt(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear8bitLt(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear8bitLt(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((1024,)

In [7]:
model_8bit_clone = copy.deepcopy(model_8bit)

lora_config = LoraConfig(
    r= 32,
    lora_alpha = 64,
    target_modules = ["q_proj","k_proj","v_proj","o_proj",
                      "gate_proj","up_proj","down_proj"],
    lora_dropout = 0.05,
    bias = "none",
    task_type = "CAUSAL_LM"
)

model_8bit_lora = get_peft_model(model_8bit_clone,lora_config)
model_8bit_lora.print_trainable_parameters()

trainable params: 20,185,088 || all params: 616,235,008 || trainable%: 3.2756


In [8]:
model_8bit_lora

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen3ForCausalLM(
      (model): Qwen3Model(
        (embed_tokens): Embedding(151936, 1024)
        (layers): ModuleList(
          (0-27): 28 x Qwen3DecoderLayer(
            (self_attn): Qwen3Attention(
              (q_proj): lora.Linear8bitLt(
                (base_layer): Linear8bitLt(in_features=1024, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1024, out_features=32, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=32, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj):

In [9]:
tokenizer = AutoTokenizer.from_pretrained(model_name,padding_side="left",trust_remote_code= True)

In [10]:
train_dataset = load_dataset("banghua/DL-SFT-Dataset")["train"]
train_dataset

Dataset({
    features: ['messages'],
    num_rows: 2961
})

In [11]:
train_dataset[0]

{'messages': [{'content': "- The left child should have a value less than the parent node's value, and the right child should have a value greater than the parent node's value.",
   'role': 'user'},
  {'content': "This statement is correct. In a binary search tree, nodes in the left subtree of a particular node have values less than the node's value, while nodes in the right subtree have values greater than the node's value. This property helps in the efficient search, insertion, and deletion of nodes in the tree.",
   'role': 'assistant'}]}

'{%- if tools %}\n    {{- \'<|im_start|>system\\n\' }}\n    {%- if messages[0].role == \'system\' %}\n        {{- messages[0].content + \'\\n\\n\' }}\n    {%- endif %}\n    {{- "# Tools\\n\\nYou may call one or more functions to assist with the user query.\\n\\nYou are provided with function signatures within <tools></tools> XML tags:\\n<tools>" }}\n    {%- for tool in tools %}\n        {{- "\\n" }}\n        {{- tool | tojson }}\n    {%- endfor %}\n    {{- "\\n</tools>\\n\\nFor each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:\\n<tool_call>\\n{\\"name\\": <function-name>, \\"arguments\\": <args-json-object>}\\n</tool_call><|im_end|>\\n" }}\n{%- else %}\n    {%- if messages[0].role == \'system\' %}\n        {{- \'<|im_start|>system\\n\' + messages[0].content + \'<|im_end|>\\n\' }}\n    {%- endif %}\n{%- endif %}\n{%- set ns = namespace(multi_step_tool=true, last_query_index=messages|length - 1) %}\n{%- for message in messa

In [15]:
def apply_chat_temp(example):
    new_text = tokenizer.apply_chat_template(example['messages'],tokenize=False)
    return {'text':new_text}

In [16]:
dataset = train_dataset.map(apply_chat_temp)
dataset

Map: 100%|██████████| 2961/2961 [00:00<00:00, 6810.73 examples/s]


Dataset({
    features: ['messages', 'text'],
    num_rows: 2961
})

In [17]:
dataset[0]

{'messages': [{'content': "- The left child should have a value less than the parent node's value, and the right child should have a value greater than the parent node's value.",
   'role': 'user'},
  {'content': "This statement is correct. In a binary search tree, nodes in the left subtree of a particular node have values less than the node's value, while nodes in the right subtree have values greater than the node's value. This property helps in the efficient search, insertion, and deletion of nodes in the tree.",
   'role': 'assistant'}],
 'text': "<|im_start|>user\n- The left child should have a value less than the parent node's value, and the right child should have a value greater than the parent node's value.<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\nThis statement is correct. In a binary search tree, nodes in the left subtree of a particular node have values less than the node's value, while nodes in the right subtree have values greater than the node's value. Th

In [18]:
def tokenize_fn(example):
    return tokenizer(example['text'])

In [20]:
tokenized_dataset = dataset.map(tokenize_fn,batched=True,remove_columns=['messages', 'text'])
tokenized_dataset

Map: 100%|██████████| 2961/2961 [00:00<00:00, 11422.60 examples/s]


Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 2961
})

In [22]:
len(tokenized_dataset['input_ids'][1])

170

In [23]:
data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False,return_tensors="pt")

In [24]:
tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token

'<|endoftext|>'

In [25]:
data_loader = DataLoader(
    tokenized_dataset,
    batch_size=2,
    collate_fn=data_collator
)

In [26]:
for step, batch in enumerate(data_loader):
    print(f"Batch {step}")
    print(batch.keys())
    print("input_ids.shape:", batch["input_ids"].shape)
    print("attention_mask.shape:", batch["attention_mask"].shape)
    print("labels.shape:", batch["labels"].shape)
    break

Batch 0
KeysView({'input_ids': tensor([[151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643,
         151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643,
         151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643,
         151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643,
         151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643,
         151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643,
         151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151644,
            872,    198,     12,    576,   2115,   1682,   1265,    614,    264,
            897,   2686,   1091,    279,   2681,   2436,    594,    897,     11,
            323,    279,   1290,   1682,   1265,    614,    264,    897,   7046,
           1091,    279,   2681,   2436,    594,    897,     13, 151645,    198,
         151644,  77091,    198, 151667,    271, 151668,    271,   1986,   511

In [30]:
training_args = TrainingArguments(
    output_dir = "./results",
    per_device_train_batch_size = 2,
    per_device_eval_batch_size = 2,
    eval_steps = 50,
    eval_strategy = "no",
    save_steps = 100,
    save_strategy = "steps",
    # num_train_epochs = 1,
    max_steps = 1000,
    learning_rate = 3e-5,
    weight_decay = 0.01,
    warmup_steps = 5,
    fp16 = False,
    bf16 = True,
    gradient_accumulation_steps = 4,
    optim = "adamw_torch",
    logging_steps = 50,
    report_to = "none"
)

In [31]:
training_args

TrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=True,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_static_graph=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=False,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=50,
eval_strategy=no,
eval_use_gather_object=Fal

In [32]:
trainer = SFTTrainer(
    model =model_8bit_lora,
    train_dataset =tokenized_dataset,
    #eval_dataset = tokenized_dataset['test'],
    args = training_args,
    data_collator= data_collator,
    processing_class = tokenizer, # in tne new version
)

In [33]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
/home/ubuntu/.local/lib/python3.10/site-packages/bitsandbytes/autograd/_functions.py:185: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Step,Training Loss
50,2.412476
100,2.143150
150,2.014931
200,1.972306
250,2.008094
300,1.961129
350,1.970331
400,1.906785
450,1.916519
500,1.881089


/home/ubuntu/.local/lib/python3.10/site-packages/bitsandbytes/autograd/_functions.py:185: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
/home/ubuntu/.local/lib/python3.10/site-packages/bitsandbytes/autograd/_functions.py:185: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
/home/ubuntu/.local/lib/python3.10/site-packages/bitsandbytes/autograd/_functions.py:185: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
/home/ubuntu/.local/lib/python3.10/site-packages/bitsandbytes/autograd/_functions.py:185: UserWarning: MatMul8bitLt: inputs will be cast

TrainOutput(global_step=1000, training_loss=1.9621775741577148, metrics={'train_runtime': 1113.5653, 'train_samples_per_second': 7.184, 'train_steps_per_second': 0.898, 'total_flos': 4211855845097472.0, 'train_loss': 1.9621775741577148})

In [34]:
trainer.model.save_pretrained('qwen_sft')

In [67]:
if not USE_GPU: # move model to CPU when GPU isn’t requested
    sft_trainer.model.to("cpu")
test_model_with_questions(trainer.model, tokenizer, questions, 
                          title="Base Model (After SFT) Output")


=== Base Model (After SFT) Output ===

Model Input 1:
Give me an 1-sentence introduction of LLM.
Model Output 1:
LLM is a large language model that can generate human-like text based on input prompts. It is used in various applications such as chatbots, content generation, and language translation. It is powered by deep learning algorithms and has been trained on vast amounts of text data. It can understand and generate natural language, making it a valuable tool for businesses and individuals alike. However, it is important to note that LLMs are not perfect and can sometimes produce inaccurate or inappropriate responses. Therefore, it is


Model Input 2:
Calculate 1+1-1
Model Output 2:
The answer is 1. The expression 1+1-1 simplifies to 2-1, which further simplifies to 1. Therefore, the result of the expression is 1.即时发生问题，请问您需要我继续计算吗？如果没有，请告诉我您需要我做什么。即时发生问题，请告诉我您需要我做什么。即时发生问题，请告诉我您需要我做什么。即时发生问题，请告诉我您需要我做什么。即时发生问题，请告诉我您需要我做什么。即时发生问题，请告诉我


Model Input 3:
What's the difference between

In this notebook we used two different models to perform SFT. At the start both models were giving absurd answers.
Post SFT both SmolLM and Qwen/Qwen3-0.6B-Base gave quite sensible answers with Qwen/Qwen3-0.6B-Base appears to give better answer that SmolLM specifically in maths exercise. Plus we also used PEFT to train Qwen/Qwen3-0.6B-Base model.